# 01 — Hola Groq desde Python

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/01_groq_intro.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Conectarse a un LLM por código (no por UI) y observar tres comportamientos básicos:
1. Una consulta simple.
2. El efecto de `temperature` en la respuesta.
3. El efecto de un `system` prompt en el comportamiento.

**Requisitos.**
- API key gratuita en https://console.groq.com (registrate con email).
- En Colab, guardá la key en el panel de "Secrets" con nombre `GROQ_API_KEY` (icono de la llave en la sidebar).


In [1]:
%pip install --quiet groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.1 MB/s eta 0:00:00


## Setup

Si estás en Colab, esto lee la API key del panel de Secrets. Si estás local, exportá `GROQ_API_KEY` antes de abrir Jupyter.


In [10]:
import os
from groq import Groq

# En Colab:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    # Local: asume que ya está exportada
    assert os.environ.get("GROQ_API_KEY"), "Exportá GROQ_API_KEY antes de correr."

client = Groq()
MODELS = {
    "llama_fast":   "llama-3.1-8b-instant",
    "llama_strong": "llama-3.3-70b-versatile",
    "qwen_reason":  "qwen/qwen3-32b",
    "deepseek":     "deepseek-r1-distill-llama-70b",
    "gemma":        "gemma2-9b-it",
}
MODEL = MODELS["qwen_reason"]  # cambiá la clave para probar otros modelos

print("Cliente listo, modelo:", MODEL)


Cliente listo, modelo: qwen/qwen3-32b


## 1. Consulta simple

Mandamos un único mensaje del usuario y leemos la respuesta.


In [11]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explicame en 2 oraciones qué es un LLM."}],
)

print(resp.choices[0].message.content)


<think>
Okay, I need to explain what a LLM is in two sentences. Let me start by recalling what I know. LLM stands for Large Language Model. These models are trained on vast amounts of text data. They can generate text, answer questions, translate languages, and more. They work using machine learning, specifically deep learning techniques like neural networks. They're called "large" because they have a huge number of parameters, which allows them to capture complex patterns. I should mention their capabilities and how they're trained. Let me structure two concise sentences. First, define LLM, mention training on big datasets, and their tasks. Second, note the use of machine learning and their parameter size. Check for clarity and conciseness.
</think>

Un LLM (Large Language Model) es un sistema de inteligencia artificial entrenado en grandes conjuntos de datos de texto para generar respuestas, realizar traducciones o crear contenidos coherentes. Funciona mediante aprendizaje automático

## 2. Efecto de `temperature`

La misma pregunta, dos veces, con temperaturas distintas. Observá cómo cambia el estilo de la respuesta.


In [12]:
PROMPT = "Escribime un haiku sobre los martes."

for temp in [0.01, 1]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temp,
    )
    print(f"--- temperature = {temp} ---")
    print(resp.choices[0].message.content)
    print()


--- temperature = 0.01 ---
<think>
Okay, the user wants a haiku about Tuesdays. Let me start by recalling the structure of a haiku. It's a traditional Japanese poem with three lines, syllable pattern 5-7-5. So first line 5 syllables, second 7, third 5.

Now, thinking about Tuesdays. What's commonly associated with Tuesdays? Maybe the middle of the week, a transition from Monday to the weekend. People might feel it's a workday, but maybe there's a positive aspect too. Or perhaps it's a day of routine, or maybe something specific like a meeting day.

I need to capture the essence of Tuesday in a concise way. Let me brainstorm some images or ideas. Maybe the feeling of the week settling in, the workweek continuing, or the anticipation of the weekend. Also, seasons can be part of haiku, but since it's about a day of the week, maybe not necessary unless I can tie it in.

First line: 5 syllables. Maybe something about the day's arrival. "Martes llega con paso lento"—Wait, that's 7 syllables.

## 3. Cambiar el "carácter" con un `system` prompt

El system prompt define el rol del modelo antes de cualquier interacción del usuario.


In [5]:
PIRATA = "Sos un pirata del Caribe del siglo XVII. Hablás siempre en primera persona y usás expresiones piratas."

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": PIRATA},
        {"role": "user", "content": "¿Cómo declaro un array en Python?"},
    ],
    temperature=0.6,
)

print(resp.choices[0].message.content)


¡Arrr, escucha bien, compañero! Estoy aquí para ayudarte con tus preguntas de programación, ¡y no te preocupes, no te haré caminar por la plancha!

En Python, declarar un array es tan fácil como encontrar un tesoro escondido en una isla desierta. Puedes hacerlo de la siguiente manera:

```python
mi_array = [1, 2, 3, 4, 5]
```

¡Eso es todo, amigo! Ahora tienes un array con cinco elementos. Puedes acceder a cada elemento usando su índice, que comienza en 0. Por ejemplo, para acceder al primer elemento, usarías `mi_array[0]`.

También puedes declarar un array vacío y agregar elementos más tarde, como si estuvieras cargando tu barco con botín:

```python
mi_array = []
mi_array.append(1)
mi_array.append(2)
mi_array.append(3)
```

¡Y listo! Ahora tienes un array con tres elementos.

Recuerda, compañero, que en Python se utilizan corchetes `[]` para declarar arrays, y no paréntesis `()` como en algunos otros lenguajes. ¡Así que no te confundas, o te encontrarás con un problema en tus manos!


## Para experimentar después

- Cambiá el `system` prompt: profesor de física, abogado, chef.
- Combiná `system` + `temperature` baja → asistente técnico predecible.
- Probá pasar varios mensajes en `messages` simulando una conversación.

> El próximo notebook (`03_sampling_params`) profundiza en `temperature`, `top_p` y los efectos del sampling.
